# Neural Minesweeper

A deep learning agent that learns to play Minesweeper from board states using PyTorch.

This project explores how neural networks can learn spatial reasoning patterns in Minesweeper by predicting mine probabilities and selecting optimal moves.

The goal is to treat Minesweeper as a machine learning problem rather than a purely logical puzzle.

## Playing by Mine Prediction

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple
import random

import numpy as np # will handle our board arrays
import matplotlib.pyplot as plt # for visualizations

import torch # used for neural network
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

Coord = Tuple[int, int] # a board cell is represented by [row, col]

**Utility Functions**<br>Small helper functions for the board navigation.

In [ ]:
def in_bounds(r: int, c: int, H: int, W: int) -> bool:
    '''checks if a position is valid on the board'''
    return 0 <= r < H and 0 <= c < W


def neighbors(cell: Coord, H: int, W: int) -> List[Coord]:
    '''returns all surrounding cells for a given square'''
    r, c = cell
    nbrs = []

    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue

            rr, cc = r + dr, c + dc
            if in_bounds(rr, cc, H, W):
                nbrs.append((rr, cc))

    return nbrs

**Enviornment result object**<br>
This stores the result of an opening cell

In [ ]:
@dataclass
class StepResult:
    '''
        - hit_mine=True if the move lost the game
        - clue=<number> if the cell was safe
        - done=True if the game is over
    '''
    hit_mine: bool
    clue: Optional[int]
    done: bool

**Minesweeper environment**<br>
This is the game engine. It creates the board, places mines, computes clues, and handles moves.

In [ ]:
class MinesweeperEnv:
    """
    Minesweeper game environment.

    Key design choices:
    - Mines are placed after the first click so the first move is always safe.
    - If force_first_zero=True, mines are also blocked from the first-click neighborhood,
      making the first clue guaranteed to be 0.
    """

    def __init__(
        self,
        H: int = 22,
        W: int = 22,
        n_mines: int = 80,
        seed: Optional[int] = None,
        force_first_zero: bool = True,
    ):
        self.H = H
        self.W = W
        self.n_mines = n_mines
        self.force_first_zero = force_first_zero
        self.rng = random.Random(seed)

        self._mines = np.zeros((H, W), dtype=bool)
        self._revealed = np.zeros((H, W), dtype=bool)
        self._clues = np.zeros((H, W), dtype=np.int32)

        self._initialized = False
        self._done = False
        self._triggered_mines = 0
        self._opened_safe = 0

    def reset(self, n_mines: Optional[int] = None) -> None:
        if n_mines is not None:
            self.n_mines = n_mines

        self._mines.fill(False)
        self._revealed.fill(False)
        self._clues.fill(0)

        self._initialized = False
        self._done = False
        self._triggered_mines = 0
        self._opened_safe = 0

    @property
    def done(self) -> bool:
        return self._done

    @property
    def opened_safe(self) -> int:
        return self._opened_safe

    @property
    def triggered_mines(self) -> int:
        return self._triggered_mines

    @property
    def mines(self) -> np.ndarray:
        return self._mines

    @property
    def revealed(self) -> np.ndarray:
        return self._revealed

    @property
    def clues(self) -> np.ndarray:
        return self._clues

    def _compute_clues(self) -> None:
        for r in range(self.H):
            for c in range(self.W):
                if self._mines[r, c]:
                    self._clues[r, c] = -1
                else:
                    mine_count = 0
                    for rr, cc in neighbors((r, c), self.H, self.W):
                        if self._mines[rr, cc]:
                            mine_count += 1
                    self._clues[r, c] = mine_count

    def _place_mines(self, first: Coord) -> None:
        forbidden: Set[Coord] = {first}
        if self.force_first_zero:
            forbidden.update(neighbors(first, self.H, self.W))

        valid_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if (r, c) not in forbidden
        ]

        if self.n_mines > len(valid_cells):
            raise ValueError("Too many mines for the board configuration.")

        sampled_mines = self.rng.sample(valid_cells, self.n_mines)
        for r, c in sampled_mines:
            self._mines[r, c] = True

        self._compute_clues()
        self._initialized = True

    def open(self, cell: Coord) -> StepResult:
        if self._done:
            return StepResult(hit_mine=False, clue=None, done=True)

        r, c = cell
        if not in_bounds(r, c, self.H, self.W):
            raise ValueError(f"Out of bounds: {cell}")

        if self._revealed[r, c]:
            return StepResult(hit_mine=False, clue=int(self._clues[r, c]), done=False)

        if not self._initialized:
            self._place_mines(first=cell)

        self._revealed[r, c] = True

        if self._mines[r, c]:
            self._triggered_mines += 1
            self._done = True
            return StepResult(hit_mine=True, clue=None, done=True)

        clue = int(self._clues[r, c])
        self._opened_safe += 1

        if self._opened_safe == (self.H * self.W - self.n_mines):
            self._done = True

        return StepResult(hit_mine=False, clue=clue, done=self._done)